## Question 3.
----

#### Writing Viterbi Algorithm for the Primer (3 marks) 

Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function `get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA")`. This should output `-41.22`. 

By doing the above two, you earn `1` mark. 

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that  maximum likely path will just be `Es`, but that is okay. Implementation is the key.

---- 


In [14]:
import math
import numpy as np

#initializing the model parameters for the Viterbi algorithm

# Define the states
states = ('E', '5', 'I')

# Define the start probabilities
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

# Define the transition probabilities
trans_prob = {
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0},
    'I': {'E': 0.1, '5': 0.0, 'I': 0.9}
}

# Define the emission probabilities for observing each nucleotide
emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}


In [15]:
## Function to Calculate Log Probability of a Given Path

def get_log_prob_of_path(state_path, sequence, start_prob, trans_prob, emit_prob):
    """
    Compute log-probability of emitting `sequence` while following
    `state_path` through an HMM defined by start_prob, trans_prob, emit_prob.
    """
    if len(state_path) != len(sequence):
        raise ValueError("state_path and sequence must have the same length")

    logp = 0.0

    s0 = state_path[0]
    
    epsilon = 1e-100
    logp += math.log(start_prob[s0] + epsilon)
    logp += math.log(emit_prob[s0][sequence[0]] + epsilon)

    for prev_state, curr_state, sym in zip(
            state_path, state_path[1:], sequence[1:]):
        logp += math.log(trans_prob[prev_state][curr_state] + epsilon)
        logp += math.log(emit_prob[curr_state][sym] + epsilon)

    return round(logp, 2)


state_path = "EEEEEEEEEEEEEEE5IIIIIIIIII"
sequence   = "CTTCATGTGAAAGCAGACGTAAGTCA"

calculated_log_prob = get_log_prob_of_path(
    state_path,
    sequence,
    start_prob,
    trans_prob,
    emit_prob
)

print(f"Log probability of path '{state_path}' for sequence '{sequence}': {calculated_log_prob}")

primer_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
primer_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA" 


primer_sequence_adjusted = primer_sequence[:len(primer_path)]

primer_log_prob = get_log_prob_of_path(
    primer_path,
    primer_sequence_adjusted,
    start_prob,
    trans_prob,
    emit_prob
)
print(f"Log probability of primer path '{primer_path}' for sequence '{primer_sequence_adjusted}': {primer_log_prob}")

Log probability of path 'EEEEEEEEEEEEEEE5IIIIIIIIII' for sequence 'CTTCATGTGAAAGCAGACGTAAGTCA': -40.28
Log probability of primer path 'EEEEEEEEEEEEEEEEEE5IIIIIII' for sequence 'CTTCATGTGAAAGCAGACGTAAGTCA': -38.92


## Viterbi Algorithm Implementation

-------

In [16]:
def viterbi(obs_seq, states, start_p, trans_p, emit_p):
    V = [{}]
    path = {} 
    T = len(obs_seq)

    def safe_log(prob):
        if prob <= 0:
            return -math.inf
        return math.log(prob)
    
    # Initialisation step
    for s in states:
        V[0][s] = safe_log(start_p.get(s, 0)) + safe_log(emit_p.get(s, {}).get(obs_seq[0], 0))
        path[s] = [s]

    # Recursion step (t = 1 to T-1)
    for t in range(1, T):
        V.append({})
        new_path = {}

        for curr_s in states:
            max_prob = -math.inf
            best_prev_s = None

            for prev_s in states:
                prob = V[t-1][prev_s] + \
                       safe_log(trans_p.get(prev_s, {}).get(curr_s, 0)) + \
                       safe_log(emit_p.get(curr_s, {}).get(obs_seq[t], 0))

                if prob > max_prob:
                    max_prob = prob
                    best_prev_s = prev_s

            V[t][curr_s] = max_prob
            if best_prev_s is not None:
                new_path[curr_s] = path[best_prev_s] + [curr_s]
            else:
                new_path[curr_s] = []


        path = new_path

    # Termination step
    max_log_prob = -math.inf
    best_final_state = None
    final_path = []

    if V[T-1]:
        for s in states:
            if s in V[T-1] and V[T-1][s] > max_log_prob:
                max_log_prob = V[T-1][s]
                best_final_state = s

    
        if best_final_state and best_final_state in path:
            final_path = path[best_final_state]
        elif V[T-1]:
             valid_paths = [p for s, p in path.items() if V[T-1].get(s, -math.inf) > -math.inf]
             if valid_paths:
                final_path = next(iter(valid_paths), [])
             else:
                 final_path = []
        else:
            final_path = []

    else:
        max_log_prob = -math.inf
        final_path = []


    return max_log_prob, final_path



observed_sequence = primer_sequence_adjusted
max_log_prob, most_likely_path = viterbi(observed_sequence, states, start_prob, trans_prob, emit_prob)

print(f"Observed sequence: {observed_sequence}")
print(f"Most likely path: {''.join(most_likely_path)}")
print(f"Log probability of the most likely path: {max_log_prob:.2f}")



Observed sequence: CTTCATGTGAAAGCAGACGTAAGTCA
Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of the most likely path: -38.68
